# 🤖 AI Tutor by Collins Aerospace
### Colab-Friendly Version (Groq + Vision RAG + Stable UI)

In [ ]:
# 1️⃣ Install Libraries
!pip install groq gradio==4.29.0 pytesseract pillow sentence-transformers faiss-cpu

In [ ]:
# 2️⃣ Imports
from groq import Groq
import gradio as gr
from PIL import Image
import pytesseract
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
# 3️⃣ Configure Groq API
client = Groq(api_key='YOUR_GROQ_API_KEY')

In [ ]:
# 4️⃣ Core LLM logic
def call_llm(system_prompt, user_prompt):
    completion = client.chat.completions.create(
        model='llama3-70b-8192',
        messages=[
            {"role":"system", "content":system_prompt},
            {"role":"user", "content":user_prompt}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
# 5️⃣ LLM Decision Engine
def decide_task(user_input):
    decision_prompt = f"Classify into chat, lesson, quiz, project: {user_input}"
    category = call_llm('classifier', decision_prompt).lower()
    if 'lesson' in category:
        return call_llm('teacher', f'Create lesson: {user_input}')
    elif 'quiz' in category:
        return call_llm('quiz', f'Generate MCQs: {user_input}')
    elif 'project' in category:
        return call_llm('mentor', f'Project ideas: {user_input}')
    else:
        return call_llm('tutor', user_input)

In [ ]:
# 6️⃣ Vision RAG Setup
embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.IndexFlatL2(384)
documents = []

In [ ]:
def extract_text(image):
    return pytesseract.image_to_string(image)

In [ ]:
def add_doc(text):
    emb = embedder.encode([text])
    index.add(np.array(emb).astype('float32'))
    documents.append(text)

In [ ]:
def retrieve(text):
    emb = embedder.encode([text])
    D, I = index.search(np.array(emb).astype('float32'), 2)
    return '\n'.join([documents[i] for i in I[0] if i < len(documents)])

In [ ]:
def evaluate(image):
    if image is None:
        return 'Upload image'
    text = extract_text(image)
    add_doc(text)
    context = retrieve(text)
    prompt = f"""
    Evaluate answer:
{text}

Context:
{context}

Give score, feedback, improvements.
"""
    return call_llm('evaluator', prompt)

In [ ]:
# 7️⃣ Colab-Safe UI (No Blocks())
def tutor_fn(text):
    return decide_task(text)

def eval_fn(img):
    return evaluate(img)

tutor_interface = gr.Interface(
    fn=tutor_fn,
    inputs=gr.Textbox(label='Ask anything'),
    outputs=gr.Textbox(label='Response', lines=10),
    title='AI Tutor - Collins Aerospace'
)

eval_interface = gr.Interface(
    fn=eval_fn,
    inputs=gr.Image(type='pil', label='Upload Answer'),
    outputs=gr.Textbox(label='Evaluation', lines=10),
    title='AI Tutor Vision Evaluation'
)

demo = gr.TabbedInterface([tutor_interface, eval_interface], ['Tutor', 'Vision Eval'])
demo.launch(share=True, debug=True)